# Decision Tree: Soil Moisture + Measurement Hour

Notebook baseline for the UTS project. Features: `soil_moisture_adc` and `jam_pengukuran`. Target: `label`.

**Important:** dry data were collected at night and wet data around midday on a different date. This creates a confounding relationship between measurement hour and label. The model may exploit time as a shortcut. Results are a baseline demonstration, not proof of generalization.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

DATA_DIR = Path("../data/raw")  # For Colab, change to the extracted project data/raw path


In [ ]:
files = ["uji_kering_1.csv", "uji_kering_2.csv", "uji_basah_1.csv", "uji_basah_2.csv"]
parts = []
for filename in files:
    part = pd.read_csv(DATA_DIR / filename)
    part["session"] = Path(filename).stem
    parts.append(part)
df = pd.concat(parts, ignore_index=True)
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum())
print("\nLabels:\n", df["label"].value_counts())
print("\nLabels per session:\n", pd.crosstab(df["session"], df["label"]))

df["recorded_at"] = pd.to_datetime(df["recorded_at"], errors="coerce")
df["soil_moisture_adc"] = pd.to_numeric(df["soil_moisture_adc"], errors="coerce")
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["jam_pengukuran"] = df["recorded_at"].dt.hour
df = df.dropna(subset=["soil_moisture_adc", "jam_pengukuran", "label"])
df = df[df["label"].isin(["kering", "basah"])].copy()

print("\nJam pengukuran per label:\n", pd.crosstab(df["jam_pengukuran"], df["label"]))
display(df.groupby("label")["soil_moisture_adc"].agg(["count","mean","median","std","min","max"]).round(2))


In [ ]:
pd.crosstab(df["jam_pengukuran"], df["label"]).plot(kind="bar")
plt.title("Measurement hour by label")
plt.xlabel("Hour of day"); plt.ylabel("Number of readings")
plt.tight_layout(); plt.show()


In [ ]:
# Features: raw ADC + hour of day. Target: dry/wet label.
X = df[["soil_moisture_adc", "jam_pengukuran"]]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, zero_division=0))
labels = ["kering", "basah"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot()
plt.title("Confusion Matrix - Two-feature baseline")
plt.show()


## Decision Tree visualization

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(model, feature_names=["soil_moisture_adc", "jam_pengukuran"],
          class_names=model.classes_, filled=True, rounded=True)
plt.tight_layout(); plt.show()


## Predict new measurements
Enter the measured ADC and the hour at which the reading was recorded. This is only a demo; because collection hour is confounded with label in the current dataset, predictions at other hours may be misleading.


In [ ]:
contoh_baru = pd.DataFrame({
    "soil_moisture_adc": [3550, 3300],
    "jam_pengukuran": [20, 13]
})
contoh_baru["prediksi"] = model.predict(contoh_baru[["soil_moisture_adc", "jam_pengukuran"]])
contoh_baru


## Interpretation and limitations
- `jam_pengukuran` is extracted from `recorded_at` with `.dt.hour`.
- `elapsed_ms` is not used as a predictor in this design.
- Dry data were collected around 20:22–21:49 on 13 October 2026; wet data were collected around 12:56–13:05 on 14 October 2026.
- Hour and label are therefore strongly confounded. The model may learn the schedule, not a general moisture relationship.
- To validate whether hour is truly useful, future data should include both dry and wet conditions at overlapping hours.
